# FOML Project — Split the Custom Image Dataset

This notebook creates **training, validation, and test sets** from our audited 198-image dataset. It does not train a model yet.

What it does:
1. Loads `processed_foml_dataset.zip` and its `metadata.csv`.
2. Checks that the image files and labels agree with the metadata.
3. Creates a reproducible **70% / 15% / 15%** split, preserving both the class label and subject category as much as possible.
4. Copies the images into `train/`, `validation/`, and `test/` folders.
5. Saves a split manifest and summary, then downloads a ZIP for the next notebook.

**Important:** Run this once and keep the resulting split fixed while comparing PCA/LDA/classical ML and CNN models. Do not make a new split for each model, because comparisons must use the same test images.


## Before running

- Upload the `processed_foml_dataset.zip` that contains `metadata.csv`, `source_audit.csv`, `README.txt`, and the processed images.
- This notebook uses `random_state = 42`, so the split is reproducible.
- With 198 images, the intended totals are **138 training, 30 validation, and 30 test images**.
- Each split will contain both labels. Category counts per split may be 2 or 3 per label/category because each category has only 16 or 17 images per label.
- The test set is reserved for final evaluation. Do not tune model settings based on test-set results.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import shutil
import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split

print("Upload your processed_foml_dataset.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Please upload exactly one dataset ZIP. ZIP files detected: {zip_names}")
ZIP_PATH = Path(zip_names[0])

RANDOM_STATE = 42
OUTPUT_DIR = Path("split_foml_dataset")
EXTRACT_DIR = Path("uploaded_processed_dataset")

# Clear only this notebook's temporary/output folders, not the uploaded ZIP.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Protect against unexpected archive paths before extracting.
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for member in zf.namelist():
        p = Path(member)
        if p.is_absolute() or ".." in p.parts:
            raise ValueError(f"Unsafe path in ZIP archive: {member}")
    zf.extractall(EXTRACT_DIR)

metadata_files = list(EXTRACT_DIR.rglob("metadata.csv"))
if len(metadata_files) != 1:
    raise ValueError(f"Expected exactly one metadata.csv, found {len(metadata_files)}")
METADATA_PATH = metadata_files[0]
DATA_ROOT = METADATA_PATH.parent

metadata = pd.read_csv(METADATA_PATH)
required_columns = {"relative_path", "label", "category"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"metadata.csv is missing columns: {sorted(missing_columns)}")

print("ZIP:", ZIP_PATH.name)
print("Dataset root:", DATA_ROOT)
print("Metadata rows:", len(metadata))
display(metadata.head())


## 1. Verify image paths and labels

We resolve every image path from the dataset folder and stop if any row points to a missing file. This avoids silently training on an incomplete dataset.

In [ ]:
# Labels/categories expected in our project.
EXPECTED_LABELS = {"real", "ai_generated"}
EXPECTED_CATEGORIES = {
    "food",
    "plants_nature",
    "buildings_and_streets",
    "household_objects",
    "vehicles",
    "animals_and_pets",
}

metadata["label"] = metadata["label"].astype(str).str.strip().str.lower()
metadata["category"] = metadata["category"].astype(str).str.strip().str.lower()

unknown_labels = sorted(set(metadata["label"]) - EXPECTED_LABELS)
unknown_categories = sorted(set(metadata["category"]) - EXPECTED_CATEGORIES)
if unknown_labels:
    raise ValueError(f"Unexpected labels: {unknown_labels}")
if unknown_categories:
    raise ValueError(f"Unexpected categories: {unknown_categories}")

# The preprocessing notebook stored paths relative to its output folder.
def resolve_image_path(relative_path):
    path = Path(str(relative_path))
    if path.is_absolute():
        return path
    return EXTRACT_DIR / path

metadata["source_path"] = metadata["relative_path"].apply(resolve_image_path)
missing_paths = metadata.loc[~metadata["source_path"].apply(Path.exists), ["relative_path", "label", "category"]]
if len(missing_paths):
    display(missing_paths.head(20))
    raise FileNotFoundError(f"{len(missing_paths)} image paths listed in metadata were not found.")

# Check the class totals and category balance before splitting.
print("Total images:", len(metadata))
print("Counts by label:")
display(metadata["label"].value_counts().rename_axis("label").to_frame("count"))
print("Counts by category and label:")
category_table = pd.crosstab(metadata["label"], metadata["category"])
display(category_table)

if len(metadata) != 198:
    print("NOTE: The number of rows is not 198. Review the counts above before continuing.")
if set(metadata["label"].unique()) != EXPECTED_LABELS:
    raise ValueError("Both 'real' and 'ai_generated' labels must be present.")
if metadata.duplicated(subset=["relative_path"]).any():
    raise ValueError("Duplicate relative paths found in metadata. Fix this before splitting.")


## 2. Create a reproducible 70/15/15 split

To keep the split fair, we stratify using a combined value such as `real_food` or `ai_generated_food`. This helps preserve both image classes and subject categories in all three sets.

We split in two stages:
- 70% training and 30% temporary data
- split the temporary data evenly into validation and test sets


In [ ]:
# Combine the class label and subject category for stratification.
metadata["strata"] = metadata["label"] + "__" + metadata["category"]

train_df, temporary_df = train_test_split(
    metadata,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=metadata["strata"],
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temporary_df["strata"],
)

train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()
train_df["split"] = "train"
validation_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)

print("Split counts:")
display(split_df["split"].value_counts().reindex(["train", "validation", "test"]).rename_axis("split").to_frame("images"))
print("Label counts within each split:")
display(pd.crosstab(split_df["split"], split_df["label"]).reindex(["train", "validation", "test"]))
print("Category counts within each split and label:")
display(split_df.groupby(["split", "label", "category"]).size().unstack(fill_value=0))

# Validate that every image appears exactly once and that all 3 splits are populated.
assert len(split_df) == len(metadata), "The number of rows changed during splitting."
assert split_df["relative_path"].nunique() == len(metadata), "An image appears more than once across splits."
assert set(split_df["split"]) == {"train", "validation", "test"}
print("\nSplit verification passed.")


## 3. Copy images into the split folders

The output uses this structure:

```text
split_foml_dataset/
  train/
    ai_generated/<category>/...
    real/<category>/...
  validation/
    ai_generated/<category>/...
    real/<category>/...
  test/
    ai_generated/<category>/...
    real/<category>/...
  split_metadata.csv
  split_summary.csv
  README.txt
```

The original processed dataset and its uploaded ZIP are left unchanged.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_rows = []

for _, row in split_df.iterrows():
    source_path = Path(row["source_path"])
    # Preserve category subfolders but keep the label folder directly under each split.
    destination = OUTPUT_DIR / row["split"] / row["label"] / row["category"] / source_path.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination)

    manifest_row = row.drop(labels=["source_path", "strata"]).to_dict()
    manifest_row["split_relative_path"] = destination.relative_to(OUTPUT_DIR).as_posix()
    manifest_rows.append(manifest_row)

split_manifest = pd.DataFrame(manifest_rows)
split_manifest.to_csv(OUTPUT_DIR / "split_metadata.csv", index=False, encoding="utf-8-sig")

summary = (
    split_manifest.groupby(["split", "label", "category"])
    .size()
    .rename("count")
    .reset_index()
)
summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")

readme = f"""FOML AI-generated vs Real Image Detection — fixed data split

Total images: {len(split_manifest)}
Training: {sum(split_manifest['split'] == 'train')}
Validation: {sum(split_manifest['split'] == 'validation')}
Test: {sum(split_manifest['split'] == 'test')}
Random state: {RANDOM_STATE}

The data was split using stratification on label + subject category. Keep this split fixed
while comparing classical ML methods (features + PCA/LDA/classifiers) with the CNN.
Only use the training set to fit models; use validation data for model choices; reserve the
test set for final evaluation after decisions are complete.

Limitations: exact-duplicate checks do not detect all near-duplicate or related images.
A good split alone cannot guarantee generalization to other image generators.
"""
(OUTPUT_DIR / "README.txt").write_text(readme, encoding="utf-8")

print("Files copied into:", OUTPUT_DIR.resolve())
print("Manifest rows:", len(split_manifest))
display(split_manifest[["image_id", "label", "category", "split", "split_relative_path"]].head())


## 4. Final checks and download

We check that each copied image exists and that the folder counts agree with the manifest before packaging the dataset.

In [ ]:
# Confirm every copied image exists.
assert split_manifest["split_relative_path"].apply(lambda p: (OUTPUT_DIR / p).is_file()).all()

# Confirm all original metadata rows received exactly one split.
assert len(split_manifest) == 198 or len(split_manifest) == len(metadata)
assert split_manifest["relative_path"].nunique() == len(metadata)
assert split_manifest["split"].value_counts().sum() == len(metadata)

print("Final image counts:")
for split_name in ["train", "validation", "test"]:
    n = int((split_manifest["split"] == split_name).sum())
    print(f"{split_name}: {n}")

print("\nLabel counts by split:")
display(pd.crosstab(split_manifest["split"], split_manifest["label"]).reindex(["train", "validation", "test"]))

OUTPUT_ZIP = Path("split_foml_dataset.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent).as_posix())

print(f"\nCreated {OUTPUT_ZIP.name} ({OUTPUT_ZIP.stat().st_size / (1024 * 1024):.1f} MB)")
files.download(str(OUTPUT_ZIP))


## What we do next

After this notebook runs successfully, keep `split_foml_dataset.zip` unchanged. Our next notebook will train and evaluate a **simple baseline CNN**. After that we will prepare a classical ML pipeline using understandable image features, PCA, LDA, and a classical classifier, then compare models on the **same fixed test set**.
